# Linear Algebra Mastery Lab

Work through this after Lessons 01–04, P2, and P5. The lab connects centering, SVD, rank, basis changes, projections, and least squares. Predict every shape before running a cell.

**Rules**: derive small cases by hand; run the assertions; then change one assumption and explain the result.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(13)
np.set_printoptions(precision=4, suppress=True)


## 1. Build a representation with hidden rank-four structure

The analysis code sees only X. The generating factors are retained solely for final verification.


In [ ]:
n, d, latent_rank = 500, 24, 4
W_true, _ = np.linalg.qr(rng.normal(size=(d, latent_rank)))
latent_scales = np.array([4.0, 2.0, 1.0, 0.5])
Z = rng.normal(size=(n, latent_rank)) * latent_scales
mean_shift = np.linspace(-3.0, 3.0, d)
X = Z @ W_true.T + 0.25 * rng.normal(size=(n, d)) + mean_shift
X_centered = X - X.mean(axis=0, keepdims=True)

assert X.shape == (n, d)
assert np.allclose(X_centered.mean(axis=0), 0.0, atol=1e-12)
print("X shape:", X.shape)


## 2. Compare raw and centered SVD

Before running: decide why a large mean shift can dominate the raw first singular direction.


In [ ]:
U_raw, s_raw, Vt_raw = np.linalg.svd(X, full_matrices=False)
U, s, Vt = np.linalg.svd(X_centered, full_matrices=False)
energy = np.cumsum(s**2) / np.sum(s**2)

assert U.shape == (n, d)
assert s.shape == (d,)
assert Vt.shape == (d, d)
assert np.allclose(U.T @ U, np.eye(d), atol=1e-10)
assert np.allclose(Vt @ Vt.T, np.eye(d), atol=1e-10)
assert np.allclose(U @ np.diag(s) @ Vt, X_centered, atol=1e-10)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(s_raw, "o-", label="raw")
ax[0].plot(s, "o-", label="centered")
ax[0].set(title="Singular values", xlabel="index", ylabel="value")
ax[0].legend()
ax[1].plot(np.arange(1, d + 1), energy, "o-")
ax[1].axhline(0.9, color="gray", linestyle="--")
ax[1].set(title="Centered cumulative energy", xlabel="rank", ylabel="fraction")
plt.tight_layout()


## 3. Truncated reconstruction and subspace recovery

The best rank-r reconstruction retains the first r singular directions. Compare reconstruction error with recovery of the true latent subspace; they are related but not identical questions.


In [ ]:
def truncated_svd_reconstruction(matrix, rank):
    u, sing, vt = np.linalg.svd(matrix, full_matrices=False)
    return (u[:, :rank] * sing[:rank]) @ vt[:rank], sing

errors = []
for rank in range(1, 11):
    approx, _ = truncated_svd_reconstruction(X_centered, rank)
    errors.append(np.linalg.norm(X_centered - approx, ord="fro"))

V_recovered = Vt[:latent_rank].T
P_recovered = V_recovered @ V_recovered.T
P_true = W_true @ W_true.T
subspace_error = np.linalg.norm(P_recovered - P_true, ord="fro")

assert all(a >= b for a, b in zip(errors, errors[1:]))
assert subspace_error < 0.25
print("rank-4 subspace error:", round(subspace_error, 4))
print("first ten reconstruction errors:", np.round(errors, 2))


## 4. Change of basis invariants

A right multiplication by an orthogonal matrix rotates coordinates. Singular values and Frobenius norms should remain unchanged.


In [ ]:
R, _ = np.linalg.qr(rng.normal(size=(d, d)))
X_rotated = X_centered @ R
s_rotated = np.linalg.svd(X_rotated, compute_uv=False)

assert np.allclose(s, s_rotated, atol=1e-10)
assert np.allclose(np.linalg.norm(X_centered, "fro"), np.linalg.norm(X_rotated, "fro"))
print("maximum singular-value change:", np.max(np.abs(s - s_rotated)))


## 5. Least-squares probe

Fit one hidden factor from the centered representation. Verify the defining orthogonality condition of least squares.


In [ ]:
target = Z[:, 0]
beta, *_ = np.linalg.lstsq(X_centered, target, rcond=None)
prediction = X_centered @ beta
residual = target - prediction
orthogonality_error = np.linalg.norm(X_centered.T @ residual)
r2 = 1 - np.sum(residual**2) / np.sum((target - target.mean())**2)

assert orthogonality_error < 1e-8
assert r2 > 0.95
print("probe R^2:", round(r2, 4))
print("residual orthogonality error:", orthogonality_error)


## Exercises

1. Choose rank using a 90%, 95%, and 99% energy threshold. Which recovers the known latent rank?
2. Increase noise from 0.25 to 1.0. Compare reconstruction and subspace error.
3. Replace the orthogonal rotation with a general invertible change of coordinates. Which quantities stop being invariant?
4. Fit the probe using rank-1 through rank-10 reconstructions. Plot predictive performance.
5. Create a rank-deficient 3×4 system, solve it with the pseudoinverse, and verify that the returned solution has minimum norm.

Carry the completed notebook into the Representation Geometry Audit capstone.
